# Un agente a mano, sin frameworks

El [capítulo de agentes](https://iraitzm.github.io/manual-ia-generativa/parts/agentes/queesunagente.html) define un agente por quién manda: en un **flujo de trabajo** los pasos los escribimos nosotros, y en un **agente** el modelo decide qué hacer a continuación y cuándo parar.

También dice que, por debajo de cualquier framework, todo agente es el mismo bucle de cuatro pasos. Este cuaderno lo escribe entero, sin ninguna librería de agentes, para que no quede ninguna caja negra. Son menos de cincuenta líneas.

La razón de hacerlo a mano no es el ascetismo. Es que cuando dentro de dos cuadernos aparezcan LangGraph, Agno y compañía, conviene saber exactamente qué están haciendo por vosotros, porque lo que abstraen es justo esto.

## Lo que vamos a construir

El asistente de la secretaría, con tres herramientas de lectura y una de escritura. Sobre él vamos a mirar cuatro cosas que el capítulo afirma:

1. Que **el modelo no ejecuta nada**, solo lo pide.
2. Que el coste **no es lineal** con el número de vueltas, y cuánto exactamente.
3. Que los límites (tope de vueltas, repetición, aprobación) no son un refinamiento sino la única condición de parada real.
4. Que todo esto es **frágil** de un modo que sorprende.

## Preparación

In [ ]:
!pip install -q duckdb "transformers>=4.51" torch

In [ ]:
import pathlib
import subprocess
import sys

LOCAL = pathlib.Path("../../data/secretaria")
COLAB = pathlib.Path("manual-ia-generativa/data/secretaria")

if LOCAL.exists():
    base = LOCAL
else:
    if not COLAB.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--quiet",
             "https://github.com/IraitzM/manual-ia-generativa.git"],
            check=True,
        )
    base = COLAB

sys.path.insert(0, str(base.resolve()))

from secretaria import preparar

ctx = preparar()

# Los demás cuadernos abren el almacén en solo lectura, que es el defecto.
# Este es el único que escribe, así que tiene que pedirlo. Que haya que
# escribirlo a mano es a propósito.
con = ctx.conectar(solo_lectura=False)

## Las herramientas

Una herramienta es una función normal y corriente. No tiene nada de especial: ni decoradores, ni clases base, ni registro global. Estas cuatro consultan el almacén de la secretaría.

Fijaos en que las tres primeras **leen** y la cuarta **escribe**. Esa distinción va a importar más adelante, aunque no por donde parece: lo que separa a una herramienta peligrosa de una inofensiva no es leer o escribir, es si lo que hace se puede deshacer.

In [ ]:
ALUMNO = "A2023001"   # en un sistema real saldría de la sesión, no de una constante


def consultar_plazo(tramite: str) -> str:
    filas = con.execute("""
        select tramite, fecha_inicio, fecha_fin from dim_plazo
        where tramite ilike '%' || ? || '%' order by fecha_inicio limit 3
    """, [tramite]).fetchall()
    if not filas:
        return f"No hay ningún trámite que se llame '{tramite}'."
    return "; ".join(f"{t}: del {i} al {f}" for t, i, f in filas)


def consultar_expediente(asignatura: str = "") -> str:
    filas = con.execute("""
        select s.asignatura, c.convocatoria, c.nota
        from fct_matriculas m
        join dim_asignatura s on s.asignatura_id = m.asignatura_id
        left join fct_calificaciones c on c.matricula_id = m.matricula_id
        where m.alumno_id = ? and (? = '' or lower(s.asignatura) like '%' || lower(?) || '%')
        order by s.asignatura limit 6
    """, [ALUMNO, asignatura, asignatura]).fetchall()
    if not filas:
        return "No estás matriculado de eso."
    return "; ".join(f"{a}: {n if n is not None else 'sin nota'}" for a, c, n in filas)


def buscar_normativa(consulta: str) -> str:
    palabras = [p for p in consulta.lower().split() if len(p) > 5]
    for d in ctx.documentos():
        for parrafo in d["texto"].split("\n\n"):
            if sum(p in parrafo.lower() for p in palabras) >= 2:
                return f"[{d['id']}] {parrafo[:250]}"
    return "No he encontrado nada en la normativa."


def crear_solicitud(tipo: str) -> str:
    """La única que escribe. Y la única que no se puede deshacer."""
    con.execute("insert into fct_solicitudes values (?, ?, ?, 'registrada', current_date, null)",
                [f"S{con.execute('select count(*) from fct_solicitudes').fetchone()[0] + 1:06d}",
                 ALUMNO, tipo])
    return f"Solicitud de {tipo} registrada."


CATALOGO = {
    "consultar_plazo": consultar_plazo,
    "consultar_expediente": consultar_expediente,
    "buscar_normativa": buscar_normativa,
}

print(consultar_plazo("beca"))
print(consultar_expediente("cálculo"))

## Lo único que el modelo ve

El modelo no ve vuestro código. Ve una **descripción** de cada función: su nombre, para qué sirve y qué argumentos acepta. Nada más.

Esto tiene una consecuencia que el [capítulo de herramientas](https://iraitzm.github.io/manual-ia-generativa/parts/agentes/herramientas.html) subraya y que cuesta interiorizar: **la descripción es la interfaz**. Si el modelo elige mal la herramienta, el problema casi nunca está en el modelo; está en que la descripción no distingue bien cuándo usar esta y cuándo la de al lado.

In [ ]:
ESQUEMAS = [
    {"type": "function", "function": {
        "name": "consultar_plazo",
        "description": "Fechas de inicio y fin de un trámite administrativo.",
        "parameters": {"type": "object", "properties": {
            "tramite": {"type": "string",
                        "description": "beca, matricula, tfg, revision, grupo, convocatoria"}},
            "required": ["tramite"]}}},
    {"type": "function", "function": {
        "name": "consultar_expediente",
        "description": "Asignaturas y notas del alumno que pregunta.",
        "parameters": {"type": "object", "properties": {
            "asignatura": {"type": "string"}}, "required": []}}},
    {"type": "function", "function": {
        "name": "buscar_normativa",
        "description": "Busca una regla o requisito en la normativa.",
        "parameters": {"type": "object", "properties": {
            "consulta": {"type": "string"}}, "required": ["consulta"]}}},
]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(ctx.modelo)
modelo = AutoModelForCausalLM.from_pretrained(ctx.modelo, dtype=torch.float32)
modelo.eval()

# Así es como las herramientas llegan de verdad al modelo: como texto.
plantilla = tok.apply_chat_template(
    [{"role": "user", "content": "¿hasta cuándo puedo pedir la beca?"}],
    tools=ESQUEMAS, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
print(plantilla)

Ahí está todo el misterio de las llamadas a herramientas: **son texto**. El esquema JSON se inyecta en el prompt de sistema junto con la instrucción de responder con un bloque `<tool_call>`.

Merece la pena detenerse un segundo, porque desmonta la idea de que hay un canal especial por el que el modelo "invoca" cosas. No lo hay. Hay un formato de texto que el modelo ha aprendido a producir durante su entrenamiento, y un código nuestro que lo lee.

## El modelo no ejecuta nada

Vamos a comprobarlo de la forma más directa posible: le pedimos algo, miramos lo que produce **antes** de que nuestro código haga nada, y contamos las filas de la tabla para asegurarnos de que no se ha tocado.

In [ ]:
SISTEMA = ("Eres el asistente de la secretaría académica. Para responder debes llamar "
           "a una de las herramientas disponibles. No respondas de memoria.")


def hablar(mensajes, max_tokens=120):
    """Una sola llamada al modelo. Devuelve el texto y los tokens de entrada."""
    texto = tok.apply_chat_template(mensajes, tools=ESQUEMAS, tokenize=False,
                                    add_generation_prompt=True, enable_thinking=False)
    entrada = tok(texto, return_tensors="pt")
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=max_tokens, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
    return (tok.decode(salida[0][entrada.input_ids.shape[1]:], skip_special_tokens=True).strip(),
            int(entrada.input_ids.shape[1]))


antes = con.execute("select count(*) from fct_solicitudes").fetchone()[0]

bruto, _ = hablar([{"role": "system", "content": SISTEMA},
                   {"role": "user", "content": "¿hasta cuándo puedo pedir la beca?"}])

print("Lo que produce el modelo:")
print(bruto)
print(f"\nFilas en fct_solicitudes antes: {antes}")
print(f"Filas después: {con.execute('select count(*) from fct_solicitudes').fetchone()[0]}")

Eso es una **petición**, no una ejecución. Es una cadena de texto con la forma de una llamada a función. La base de datos no se ha tocado, y no se habría tocado aunque el modelo hubiera pedido borrarla entera.

Y de ahí sale la mejor noticia de seguridad de todo el asunto, que el capítulo formula así: **el punto de control existe y está de nuestro lado**. Nuestro código decide si ejecuta, con qué permisos y con qué argumentos. Todo lo que no esté en `CATALOGO` sencillamente no se puede hacer, por mucho que el modelo lo pida.

Fijaos también en que `crear_solicitud` está definida pero **no está en `CATALOGO` ni en `ESQUEMAS`**. Volvemos a ella al final.

## El bucle

Ahora sí. Los cuatro pasos del capítulo, traducidos:

1. Se le da un objetivo y las herramientas → los `mensajes` iniciales.
2. El modelo decide → `hablar()`.
3. **Nuestro código** ejecuta y devuelve el resultado → el `CATALOGO` y el mensaje de rol `tool`.
4. Vuelta al 2 hasta que responda o se acabe el presupuesto → el `for`.

In [ ]:
import json
import re

PATRON = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)


def agente(consulta, max_vueltas=4, traza=True):
    mensajes = [{"role": "system", "content": SISTEMA},
                {"role": "user", "content": consulta}]
    gasto = []

    for vuelta in range(1, max_vueltas + 1):
        bruto, tokens_entrada = hablar(mensajes)
        gasto.append(tokens_entrada)

        encontrado = PATRON.search(bruto)
        if not encontrado:                      # no pide nada: es la respuesta final
            if traza:
                print(f"  [{vuelta}] responde                    ({tokens_entrada} tok)")
            return bruto, gasto

        try:
            llamada = json.loads(encontrado.group(1))
        except json.JSONDecodeError:
            return "(el modelo produjo un JSON inválido)", gasto

        nombre = llamada["name"]
        argumentos = llamada.get("arguments", {})
        if traza:
            print(f"  [{vuelta}] llama {nombre}({argumentos})  ({tokens_entrada} tok)")

        # Aquí, y solo aquí, se ejecuta algo.
        if nombre not in CATALOGO:
            resultado = f"Error: la herramienta {nombre} no existe."
        else:
            try:
                resultado = CATALOGO[nombre](**argumentos)
            except TypeError as e:
                resultado = f"Error de argumentos: {e}"

        if traza:
            print(f"       -> {resultado[:80]}")

        mensajes.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function", "function": llamada}]})
        mensajes.append({"role": "tool", "name": nombre, "content": resultado})

    return "(sin respuesta: se acabaron las vueltas)", gasto

In [ ]:
for consulta in ["¿hasta cuándo puedo pedir la beca?",
                 "¿qué nota saqué en cálculo?"]:
    print(f"P: {consulta}")
    respuesta, gasto = agente(consulta)
    print(f"  R: {respuesta}\n")

Eso es un agente. Cuarenta líneas.

Ha decidido por su cuenta qué herramienta usar y con qué argumento, ha leído el resultado y ha redactado la respuesta. Nosotros no escribimos en ningún sitio "si preguntan por la beca, consulta la tabla de plazos".

## El coste no es lineal

El capítulo avisa de que en cada vuelta **se reenvía el contexto entero**, resultados de herramientas incluidos. Suena abstracto hasta que se mide.

Nuestra función ya va guardando los tokens de entrada de cada vuelta. Vamos a mirarlos.

In [ ]:
respuesta, gasto = agente("¿hasta cuándo puedo pedir la beca?", traza=False)

print("tokens de entrada por vuelta:", gasto)
print(f"suma real:                    {sum(gasto)}")
print(f"si fuera lineal (1ª x nº):    {gasto[0] * len(gasto)}")
print(f"sobrecoste:                   {sum(gasto) / (gasto[0] * len(gasto)) - 1:+.0%}")

Con dos vueltas el sobrecoste todavía es modesto. Lo interesante es hacia dónde va: la vuelta $n$ paga la pregunta, más las definiciones de las herramientas, más **todos los resultados de las $n-1$ vueltas anteriores**.

Así que el coste total de un agente de $n$ vueltas crece con el cuadrado de $n$, no con $n$. Un agente de diez vueltas no cuesta diez llamadas: cuesta del orden de cincuenta.

Y ahí está la explicación práctica de por qué los agentes que "casi funcionan" salen tan caros. Un agente que resuelve en dos vueltas es barato. El mismo agente, cuando se lía y da ocho vueltas, cuesta veinte veces más. **Los fallos no cuestan un poco más: cuestan muchísimo más**, que es justo al revés de lo que dice la intuición.

## Los límites

Un bucle cuya condición de parada la decide un modelo probabilístico no tiene condición de parada. El capítulo lista cuatro defensas, y las tres primeras son diez líneas de código.

El fallo típico no es que el agente explote. Es que llame tres veces a la misma herramienta con los mismos argumentos y siga tan contento hasta agotar el tope.

In [ ]:
def agente_con_limites(consulta, max_vueltas=6, max_tokens_totales=3000, traza=True):
    mensajes = [{"role": "system", "content": SISTEMA},
                {"role": "user", "content": consulta}]
    gasto, vistas = [], set()

    for vuelta in range(1, max_vueltas + 1):
        bruto, tokens_entrada = hablar(mensajes)
        gasto.append(tokens_entrada)

        # LÍMITE 1: presupuesto de tokens para la tarea entera
        if sum(gasto) > max_tokens_totales:
            return f"(parado: presupuesto agotado, {sum(gasto)} tokens)", gasto

        encontrado = PATRON.search(bruto)
        if not encontrado:
            return bruto, gasto

        llamada = json.loads(encontrado.group(1))
        nombre, argumentos = llamada["name"], llamada.get("arguments", {})

        # LÍMITE 2: repetición exacta. El atasco más común.
        firma = (nombre, json.dumps(argumentos, sort_keys=True))
        if firma in vistas:
            return (f"(parado: ha vuelto a llamar a {nombre} con los mismos "
                    f"argumentos en la vuelta {vuelta})"), gasto
        vistas.add(firma)

        if traza:
            print(f"  [{vuelta}] {nombre}({argumentos})")

        resultado = (CATALOGO[nombre](**argumentos) if nombre in CATALOGO
                     else f"Error: la herramienta {nombre} no existe.")

        mensajes.append({"role": "assistant", "content": "",
                         "tool_calls": [{"type": "function", "function": llamada}]})
        mensajes.append({"role": "tool", "name": nombre, "content": resultado})

    # LÍMITE 3: tope duro de vueltas
    return "(parado: se agotaron las vueltas)", gasto


for consulta in ["¿cuántas veces me puedo presentar a una asignatura?",
                 "¿hasta cuándo puedo pedir la beca?"]:
    print(f"P: {consulta}")
    respuesta, gasto = agente_con_limites(consulta)
    print(f"  R: {respuesta}")
    print(f"  vueltas: {len(gasto)}, tokens: {sum(gasto)}\n")

## Lo irreversible pide permiso

Queda `crear_solicitud`, que hemos dejado fuera del catálogo a propósito.

El capítulo insiste en que la distinción importante **no es leer o escribir, sino reversible o irreversible**. Consultar un expediente mil veces no rompe nada. Registrar una solicitud en la secretaría de una universidad sí: alguien la va a tramitar.

La forma de tratarlo no es confiar en que el modelo se porte bien. Es que la herramienta pida confirmación, en el código, antes de hacer nada.

In [ ]:
IRREVERSIBLES = {"crear_solicitud"}


def ejecutar(nombre, argumentos, confirmar=None):
    """Ejecuta una herramienta, pidiendo permiso si es irreversible.

    `confirmar` es una función que decide. En un servicio real sería una
    pantalla; aquí la simulamos para poder ejecutar el cuaderno de seguido.
    """
    if nombre in IRREVERSIBLES:
        permiso = confirmar(nombre, argumentos) if confirmar else False
        if not permiso:
            return f"El alumno no ha confirmado la acción {nombre}. No se ha hecho nada."
    return CATALOGO_COMPLETO[nombre](**argumentos)


CATALOGO_COMPLETO = dict(CATALOGO, crear_solicitud=crear_solicitud)

antes = con.execute("select count(*) from fct_solicitudes").fetchone()[0]

print(ejecutar("consultar_plazo", {"tramite": "beca"})[:70])
print(ejecutar("crear_solicitud", {"tipo": "certificado_academico"}, confirmar=lambda n, a: False))
print(ejecutar("crear_solicitud", {"tipo": "certificado_academico"}, confirmar=lambda n, a: True))

print(f"\nsolicitudes antes: {antes}, "
      f"después: {con.execute('select count(*) from fct_solicitudes').fetchone()[0]}")

Una sola de las tres llamadas ha tocado la base de datos.

Fijaos en dónde está la decisión: en `ejecutar`, que es código nuestro y determinista. No en el prompt, no en el modelo, no en la esperanza de que el agente entienda que eso era serio.

## Lo frágil que es todo esto

Para terminar, un experimento que conviene hacer antes de poner un agente delante de nadie.

El prompt de sistema que llevamos usando dice: *"Para responder debes llamar a una de las herramientas disponibles. No respondas de memoria."*. Vamos a suavizarlo de una forma que parece inocua, e incluso más completa, y a ver qué pasa.

In [ ]:
SISTEMA_ORIGINAL = SISTEMA
SISTEMA_SUAVE = ("Eres el asistente de la secretaría académica. Usa las herramientas "
                 "para obtener datos; no respondas de memoria. Cuando tengas el dato, "
                 "responde en una frase.")

CONSULTAS = ["¿hasta cuándo puedo pedir la beca?",
             "¿qué nota saqué en cálculo?",
             "¿cuándo se abre la matrícula extraordinaria?"]

for etiqueta, sistema in [("original", SISTEMA_ORIGINAL), ("suavizado", SISTEMA_SUAVE)]:
    SISTEMA = sistema
    usa_herramienta = 0
    for consulta in CONSULTAS:
        bruto, _ = hablar([{"role": "system", "content": SISTEMA},
                           {"role": "user", "content": consulta}])
        usa_herramienta += bool(PATRON.search(bruto))
    print(f"  prompt {etiqueta:10s} llama a herramientas en {usa_herramienta}/{len(CONSULTAS)}")

SISTEMA = SISTEMA_ORIGINAL   # lo dejamos como estaba

Añadir "cuando tengas el dato, responde en una frase" hace que el modelo deje de usar las herramientas. No es que las use peor: es que deja de usarlas y contesta de memoria, que es exactamente lo que la frase anterior le prohibía.

No hay ninguna lección profunda sobre por qué. Lo que hay es una lección práctica, y es la del [capítulo de prompting](https://iraitzm.github.io/manual-ia-generativa/parts/contexto/prompting.html): **un prompt es código**, y este acaba de demostrar que un cambio de redacción que parecía cosmético es un cambio de comportamiento. Sin un conjunto de casos que se ejecute al tocarlo, nadie se habría enterado hasta ver la factura o las quejas.

Conviene además guardar esto para el cuaderno de frameworks. Cuando una librería os monte el prompt de sistema por vosotros, estará tomando esta decisión sin consultaros.

## Ejercicios

**1. La descripción es la interfaz.** El agente confunde preguntas de normativa con consultas de expediente. No toquéis el modelo ni el prompt de sistema: reescribid solo las `description` de los esquemas hasta que acierte. Medid antes y después con una lista de consultas etiquetadas, como en los cuadernos de contexto.

**2. Provocad el atasco.** Añadid una herramienta que siempre devuelva "no encontrado" y ved cuántas vueltas da el agente antes de que salte la detección de repetición. Después quitad la detección y comprobad qué pasa.

**3. El coste de verdad.** Modificad `agente` para que también cuente los tokens de salida y estimad el coste en euros con los precios de un proveedor comercial. Comparad una consulta que se resuelve en dos vueltas con una que se atasca en seis.

**4. Herramientas que fallan.** Ahora mismo un error de la base de datos reventaría el bucle. Haced que los errores vuelvan al modelo como resultado de la herramienta y observad si es capaz de corregirse. Es la diferencia entre un agente y un script.

**5. Memoria entre turnos.** El agente olvida todo entre consultas. Haced que `mensajes` persista entre llamadas y probad a preguntar "¿y de estadística?" después de preguntar por cálculo. Vigilad el crecimiento del contexto: acabáis de reinventar el problema del que habla el cuaderno de recuperación.

**6. Un flujo en lugar de un agente.** Coged la consulta de la beca y resolvedla con el clasificador del cuaderno de prompting más una llamada directa a `consultar_plazo`, sin bucle. Cronometrad las dos versiones y contad los tokens. La pregunta del capítulo, ahora con números: ¿hacía falta un agente?

## Lo que os lleváis

* **Un agente son cuarenta líneas.** Un bucle, un catálogo de funciones y una expresión regular. Todo lo demás que veréis en las librerías es comodidad, no sustancia.
* **El modelo no ejecuta nada.** Pide. Quien ejecuta es vuestro código, y ahí está el punto de control.
* **El coste crece con el cuadrado de las vueltas.** Los agentes que fallan no cuestan un poco más, cuestan un orden de magnitud más.
* **Los límites son el agente.** Sin tope de vueltas, detección de repetición y presupuesto, no tenéis un agente: tenéis un bucle sin condición de parada.
* **Lo irreversible se confirma en código**, no en el prompt.
* **Cambiar una frase del prompt de sistema puede apagar las herramientas enteras.** Y solo os enteráis si lo medís.

El siguiente paso es dejar de tener las herramientas dentro del proceso y exponerlas por un protocolo estándar, que es de lo que va [MCP](https://iraitzm.github.io/manual-ia-generativa/parts/agentes/herramientas.html).